In [3]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanumsearch-embs")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [4]:
import numpy as np

def parse_fast(example):
    arr = np.fromstring(example["embedding"].strip("[]"), sep=", ", dtype=np.float32)
    example["embedding"] = arr.tolist()
    return example

dataset = dataset.map(parse_fast)

In [ ]:
from sentence_transformers import (
    SentenceTransformer,
)

model = SentenceTransformer('pameydorke/arcanum-cross-platform-retriever')

In [5]:
import torch

embeddings = torch.tensor(dataset["train"]["embedding"], dtype=torch.float32)

In [ ]:
import torch.nn.functional as F

def retrieve(query, top_k=5):
    query_embs = model.encode(query, convert_to_tensor=True)

    similarities = F.cosine_similarity(query_embs, embeddings)

    scores, indices = torch.topk(similarities, k=top_k)

    return [
        {"score": float(scores[i]), "link": dataset["train"][indices[i]]["url"], "text": dataset["train"][indices[i]]["content"]}
        for i in range(top_k)
    ]

tensor([ 1.0000, -0.0854,  0.0782,  ...,  0.2379,  0.1536, -0.0668])

In [ ]:
# TODO: recall, ndcg

In [6]:
sample_dataset = dataset["train"].shuffle(seed=42).select(range(100))

In [17]:
i = 10

sample_dataset[i]["url"]

'https://reddit.com/r/arcanum/comments/1apyooq/starting_game_for_first_time_but_having_trouble/'